# Colab T4 GPU — Free NVIDIA T4 Plugin

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/romangalaxys10-spec/ai-agent-architect/blob/main/agents/colab-t4-gpu-agent/notebook/colab_t4_gpu.ipynb)

**Runtime → Change runtime type → T4 GPU → Save → Runtime → Restart runtime** before running. Free T4 = 16GB VRAM, ~8.1 TFLOPS.


In [ ]:
# 0) Detect T4
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch, platform
print(f"Colab={__import__('os').environ.get('COLAB_RELEASE_TAG', 'no')[:20]}")
print(f"Python {platform.python_version()}  torch {torch.__version__}")
print(f"cuda={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(f"VRAM {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")


In [ ]:
# 1) One-shot setup (idempotent)
%pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121 accelerate bitsandbytes transformers
# optional (soft-fail on T4):
# %pip install -q xformers
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no-cuda")


In [ ]:
# 2) Benchmark — VRAM + TTFT/TPS (T4-aware routing: ≤7B 4-bit, 13B GGUF, 70B API)
import time, torch
from plugin.colab_t4_plugin import benchmark_t4
print(benchmark_t4())
# Tiny live inference (fits T4 16GB in 4-bit):
from transformers import AutoModelForCausalLM, AutoTokenizer
model_id="Qwen/Qwen2-0.5B-Instruct"
tok=AutoTokenizer.from_pretrained(model_id)
mdl=AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", load_in_4bit=True)
prompt="Explain Colab T4 free GPU in one sentence."
ids=tok(prompt, return_tensors="pt").to(mdl.device)
t0=time.time()
out=mdl.generate(**ids, max_new_tokens=64)
print(f"TTFT {time.time()-t0:.2f}s")
print(tok.decode(out[0], skip_special_tokens=True))


### CLI (same engine, local + Colab)
```bash
!python -m agents.colab_t4_gpu_agent.cli.colab_t4 --check
!python -m agents.colab_t4_gpu_agent.cli.colab_t4 --setup --dry-run
!python -m agents.colab_t4_gpu_agent.cli.colab_t4 --benchmark
```
